# 🚌 대구버스 공공데이터 API 실습 워크시트

**목표:** 공공데이터포털 API 문서를 읽고, 직접 데이터를 불러와봅니다.

---
## ✅ 시작 전 체크리스트
- [ ] 공공데이터포털 회원가입 완료
- [ ] 대구버스정보시스템 API 활용 신청 완료
- [ ] API 키 승인 완료 (마이페이지 > 활용신청 현황에서 확인)

> 📌 API 페이지 주소: https://www.data.go.kr/data/15139134/openapi.do

---
## 📖 Step 0. API가 뭔가요?

**API(Application Programming Interface)** 는 프로그램끼리 데이터를 주고받는 약속된 방법입니다.

```
🙋 나(파이썬 코드)
      ↓ requests.get() 으로 요청
🏢 공공데이터 서버
      ↓ JSON 데이터로 응답
📦 데이터 수신!
```

### 응답 코드 의미
| 코드 | 의미 |
|------|------|
| **200** | ✅ 성공! |
| **401** | 🔴 API 키 오류 |
| **400** | 🔴 파라미터 오류 |
| **500** | 🔴 서버 오류 |

---
## 🔑 Step 1. API 키 설정

공공데이터포털 마이페이지에서 발급받은 **일반 인증키(Decoding)** 를 입력하세요.

> ⚠️ 주의: 키 앞뒤에 공백이나 탭이 들어가면 401 오류가 납니다! `.strip()` 필수!

In [36]:
# 비동기(async) 허용
# --> jupyter notebook에서 이미 허용
# --> 라이브러리 실행 또는 다양한 상황에서 vs code가 또 허용
# --> 이미 돌고 있는 또 돌아라 --> 중첩 (멘붕) --> 허용
import nest_asyncio
nest_asyncio.apply()

In [37]:
import os
from dotenv import load_dotenv

load_dotenv()

API_KEY = os.getenv('OPENAPI_API_KEY')

In [41]:
import requests
import json

# 🔑 여기에 본인의 API 키를 입력하세요
# API_KEY = b8e20e31222eca5291651106c0ff043cc822ca36b042a6095828e2db6f9e0a1d.strip()

print("API 키 길이:", len(API_KEY))
print("앞 10자리:", API_KEY[:10], "...")

API 키 길이: 72
앞 10자리: b8e20e3122 ...


---
## 📡 Step 2. API 문서 읽기 연습

아래 사이트에서 **getBasic02** 항목을 찾아 다음을 채워보세요.

🌐 https://www.data.go.kr/data/15139134/openapi.do

### 📝 탐색 미션 (빈칸 채우기)

```
1. API 이름: getBasic02
2. 기능 설명: 노선, 정류장, 노드, 링크를 제공받는 서비스
3. 필수 파라미터(*): serviceKey
4. 응답 데이터에 포함된 항목들:
   - route (노선): routeId, routeNo, dataconnareacd, stBsId, edBsId, stNm, edNm, routeNote, dirRouteNote, ndirRouteNote, routeTCd
   - bs (정류장): bsId, bsNm(정류장 이름), wincId(모바일ID), xPos, yPos
   - node (노드): nodeId, nodeNm, xPos, yPos, bsYn
```

---
## 🚀 Step 3. getBasic02 호출하기 (전체 기초 데이터)

In [46]:
# 📡 API 주소
url = "https://apis.data.go.kr/6270000/dbmsapi02/getBasic02"

# 📋 요청 파라미터
params = {
    "serviceKey": API_KEY
}

# 🚀 요청 보내기
response = requests.get(url, params=params)

# ✅ 결과 확인
print("응답 코드:", response.status_code)

if response.status_code == 200:
    print("✅ 성공!")
elif response.status_code == 401:
    print("🔴 API 키를 확인하세요!")
else:
    print("🔴 오류:", response.status_code)

응답 코드: 401
🔴 API 키를 확인하세요!


---
## 🔍 Step 4. 응답 데이터 구조 탐색

In [39]:
# JSON으로 변환
data = response.json()

# 전체 키 확인
# print(data)
# print('-' * 50)
print("최상위 키:", list(data.keys()))

JSONDecodeError: Expecting value: line 1 column 1 (char 0)

In [ ]:
# 헤더 확인 (성공 여부)
header = data["header"]
print("성공 여부:", header["success"])
print("결과 메시지:", header["resultMsg"])
print("결과 코드:", header["resultCode"])

NameError: name 'data' is not defined

In [ ]:
# 바디 확인 (실제 데이터)
body = data["body"]
print("총 데이터 수:", body["totalCount"])
print("items 키 목록:", list(body["items"].keys()))

---
## 🚌 Step 5. 노선 정보 꺼내기

In [ ]:
# 노선 데이터
routes = data["body"]["items"]["route"]

print(f"전체 노선 수: {len(routes)}개")
print("\n첫 번째 노선 정보:")
print(json.dumps(routes[0], ensure_ascii=False, indent=2))

In [ ]:
# 노선 번호만 출력 (앞 10개)
print("노선 번호 목록 (앞 10개):")
for route in routes[:10]:
    print(f"  - {route['routeNo']} ({route['stNm']} → {route['edNm']})")

### 🎯 미션 1
아래 코드를 완성해서 **급행** 노선만 출력해보세요!

In [ ]:
# 🎯 미션 1: 급행 노선만 찾기
print("급행 노선 목록:")
for route in routes:
    if "급행" in route["routeNo"]:   # ← 이 조건을 바꿔보세요!
        print(f"  🚌 {route['routeNo']} | {route['stNm']} → {route['edNm']}")

---
## 🚏 Step 6. 정류장 정보 꺼내기

In [ ]:
# 정류장 데이터
stops = data["body"]["items"]["bs"]

print(f"전체 정류장 수: {len(stops)}개")
print("\n첫 번째 정류장 정보:")
print(json.dumps(stops[0], ensure_ascii=False, indent=2))

### 🎯 미션 2
아래 빈칸을 채워서 **"동대구역"** 이 이름에 포함된 정류장을 모두 출력해보세요!

In [ ]:
# 🎯 미션 2: 동대구역 정류장 찾기
keyword = "동대구역"   # ← 원하는 키워드로 바꿔보세요

print(f"'{keyword}' 포함 정류장:")
for stop in stops:
    if keyword in stop["___"]:   # ← 빈칸: 정류장 이름 필드명은?
        print(f"  🚏 {stop['bsNm']} | ID: {stop['bsId']}")

---
## 🔗 Step 7. 다른 API 탐색하기

API 문서 페이지에서 다른 API들을 찾아 탐색해봅시다.

### 📝 탐색 미션 (빈칸 채우기)

| API 이름 | 기능 | 필수 파라미터 | 내가 뽑고 싶은 데이터 |
|----------|------|--------------|---------------------|
| getBasic02 | 기초종합 정보 | serviceKey | 노선/정류장 목록 |
| getBs02 |경유하는 정류장 정보제공하는 서비스 | serviceKey, routeId  | |
| getLink02 |경유하는 링크 정보제공하는 서비스 | serviceKey, routeId  | |
| getRealtime02 | 도착예정정보를 제공하는 서비스 | serviceKey, bsId, routeNo | |
| getPos02 | 버스 위치 정보제공하는 서비스 | serviceKey, routeId  | |

In [44]:
# 🎯 미션 3: getBs02 - 정류장 상세 정보 조회
# Step 6에서 찾은 정류장 ID를 활용해보세요!

url_bs = "https://apis.data.go.kr/6270000/dbmsapi02/getBs02"

params_bs = {
    "serviceKey": API_KEY,
    "busStopId": "여기에_정류장ID_입력"   # ← Step 6에서 찾은 bsId 입력
}

response_bs = requests.get(url_bs, params=params_bs)
print("응답 코드:", response_bs.status_code)

if response_bs.status_code == 200:
    data_bs = response_bs.json()
    print(json.dumps(data_bs, ensure_ascii=False, indent=2))

응답 코드: 401


---
## 🏆 Step 8. 나만의 탐색 코드 작성

지금까지 배운 것을 활용해서 **내가 원하는 정보를 조회하는 코드**를 작성해보세요!

**아이디어 예시:**
- 집 근처 정류장 찾기
- 특정 번호 버스의 출발지/도착지 확인
- 정류장 이름에 '학교'가 들어가는 곳 찾기

In [ ]:
# 🏆 나만의 탐색 코드
# 아이디어: ___________________________

# 여기에 코드를 작성하세요!


---
## 📚 정리

오늘 배운 것:

| 개념 | 코드 | 설명 |
|------|------|------|
| API 요청 | `requests.get(url, params=params)` | 서버에 데이터 요청 |
| 응답 확인 | `response.status_code` | 200이면 성공 |
| JSON 변환 | `response.json()` | 딕셔너리로 변환 |
| 데이터 접근 | `data["body"]["items"]` | 키로 접근 |
| 데이터 탐색 | `for item in items:` | 반복문으로 탐색 |
| 조건 필터 | `if "키워드" in item["필드"]:` | 원하는 것만 추출 |
